# Data Quality and the Write-Audit-Publish Pattern

The gold layer is what analysts, dashboards, and ML models consume. A single bad row in gold can propagate into wrong decisions across the entire organization. This notebook covers how to validate gold data before publishing it, and how Iceberg branches enable a clean Write-Audit-Publish gate that leaves production data untouched until quality checks pass.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import json
import shutil
from pathlib import Path
import pyarrow as pa
import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog
from pyspark.sql import functions as F

sys.path.insert(0, str(Path.cwd()))

In [ ]:
for d in ["../data/warehouse_silver", "../data/warehouse_gold"]:
    shutil.rmtree(d, ignore_errors=True)

catalog = SqlCatalog("silver", **{
    "uri": "sqlite:///../data/warehouse_silver/silver_catalog.db",
    "warehouse": "file://" + str(Path("../data/warehouse_silver").resolve()),
})
catalog.create_namespace_if_not_exists("silver")

events = [json.loads(l) for l in open("../data/input/events.jsonl")]
arrow_table = pa.Table.from_pandas(pd.DataFrame(events))
silver_events = catalog.create_table("silver.events", schema=arrow_table.schema)
silver_events.append(arrow_table)
print(f"Silver: {len(events):,} events")

In [ ]:
from helpers import get_spark

spark = get_spark("DataQuality", gold_warehouse="../data/warehouse_gold")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")

# Build hourly gold table
silver_df = spark.createDataFrame(silver_events.scan().to_arrow().to_pandas())
silver_df = silver_df.withColumn("event_hour", F.date_trunc("hour", F.to_timestamp("time")))

hourly_gold = silver_df.groupBy("event_hour", "type", "source").agg(
    F.count("*").alias("event_count")
).orderBy("event_hour")

hourly_gold.writeTo("local.gold.device_hourly").createOrReplace()
print(f"Gold rows: {hourly_gold.count():,}")

## Why Data Quality Matters Most at Gold

Data quality has five key dimensions at the gold layer:

- **Completeness**: are required fields present? A device_id NULL in gold means lost data.
- **Accuracy**: do values fall in plausible ranges? event_count < 0 is impossible.
- **Consistency**: does every device in gold exist in the device registry (cmdata.jsonl)?
- **Uniqueness**: is the same event counted twice? IoT devices retry on failure, creating duplicates.
- **Timeliness**: is the data fresh? A gold table last updated 6 hours ago may not meet SLAs.

Silver has raw noise — that's expected. Gold is what people trust. Bad gold multiplies into wrong decisions.

## Quality Checks in Spark

Quality checks are just SQL queries or PySpark assertions:

- A check passes if it returns zero rows (no violations) or meets a threshold.
- Encode checks as functions that raise an exception on failure — so the pipeline fails loudly instead of silently writing bad data.

This makes quality gates composable: you can run a list of check functions and collect all failures before aborting.

In [ ]:
def check_no_nulls(spark, table, columns):
    """Raise if any row has a NULL in the specified columns."""
    for col in columns:
        null_count = spark.sql(f"SELECT COUNT(*) as n FROM {table} WHERE {col} IS NULL").collect()[0]["n"]
        if null_count > 0:
            raise ValueError(f"Quality check failed: {null_count} NULL values in {table}.{col}")
    print(f"Null check passed for {columns} in {table}")

check_no_nulls(spark, "local.gold.device_hourly", ["event_hour", "type", "source", "event_count"])

In [ ]:
def check_non_negative_counts(spark, table, count_columns):
    """Raise if any count column has a negative value."""
    for col in count_columns:
        neg_count = spark.sql(f"SELECT COUNT(*) as n FROM {table} WHERE {col} < 0").collect()[0]["n"]
        if neg_count > 0:
            raise ValueError(f"Quality check failed: {neg_count} negative values in {table}.{col}")
    print(f"Range check passed for {count_columns} in {table}")

check_non_negative_counts(spark, "local.gold.device_hourly", ["event_count"])

In [ ]:
# Load device registry
devices = [json.loads(l) for l in open("../data/input/cmdata.jsonl")]
valid_ids = {str(d["id"]) for d in devices}
valid_ids_df = spark.createDataFrame([(id_,) for id_ in valid_ids], ["device_id"])

# Check: find device IDs in gold that don't appear in cmdata
gold_devices = spark.sql("SELECT DISTINCT source FROM local.gold.device_hourly")
unknown_devices = gold_devices.join(valid_ids_df, gold_devices["source"] == valid_ids_df["device_id"], "left_anti")
unknown_count = unknown_devices.count()
print(f"Devices in gold not found in cmdata: {unknown_count:,}")
# Note: this is informational — some devices may be valid but not appear in our local cmdata.jsonl

In [ ]:
def check_unique_key(spark, table, key_columns):
    """Raise if the key columns are not unique."""
    key_expr = ", ".join(key_columns)
    dup_count = spark.sql(f"""
        SELECT COUNT(*) as n FROM (
            SELECT {key_expr}, COUNT(*) as cnt
            FROM {table}
            GROUP BY {key_expr}
            HAVING cnt > 1
        )
    """).collect()[0]["n"]
    if dup_count > 0:
        raise ValueError(f"Quality check failed: {dup_count} duplicate key combinations in {table}")
    print(f"Uniqueness check passed for {key_columns} in {table}")

check_unique_key(spark, "local.gold.device_hourly", ["event_hour", "type", "source"])

In [ ]:
# Flag hours with event counts > 3 standard deviations above the mean
stats = spark.sql("""
    SELECT
        AVG(event_count) AS mean_count,
        STDDEV(event_count) AS std_count
    FROM local.gold.device_hourly
""").collect()[0]

mean_count = stats["mean_count"]
std_count = stats["std_count"]
threshold = mean_count + 3 * std_count

anomalies = spark.sql(f"""
    SELECT event_hour, type, source, event_count
    FROM local.gold.device_hourly
    WHERE event_count > {threshold:.2f}
    ORDER BY event_count DESC
    LIMIT 10
""")

print(f"Mean: {mean_count:.1f}, Std: {std_count:.1f}, Threshold (3\u03c3): {threshold:.1f}")
print(f"Anomalous rows: {anomalies.count()}")
anomalies.show()

## The Write-Audit-Publish Pattern

The traditional approach: write gold data → run quality checks → if they fail, delete the bad data. The problem is that the window between write and check leaves bad data visible to consumers.

Write-Audit-Publish (WAP) solves this by validating data **before** it reaches the main table:

1. **Write**: compute the new gold data in a staging DataFrame (or a separate staging table).
2. **Audit**: run quality checks against the staged data. If they fail, discard it — the main table is never touched.
3. **Publish**: if checks pass, commit the data to the main gold table.

In production Iceberg setups, step 1 uses a branch: writes go to an isolated branch of the gold table, so even if you forget to audit, consumers reading `main` see nothing new. Branches are lightweight (just a pointer to a snapshot ID) and abandoning one is a metadata-only operation.

For this demo we use the pre-write validation pattern, which is equivalent from a data quality perspective.

In [ ]:
# Build a new gold batch (simulate next pipeline run) with deliberate quality issues
import datetime
from pyspark.sql.types import StructType, StructField, TimestampType, StringType, LongType

bad_schema = StructType([
    StructField("event_hour", TimestampType(), nullable=True),
    StructField("type", StringType(), nullable=False),
    StructField("source", StringType(), nullable=False),
    StructField("event_count", LongType(), nullable=False),
])

new_gold_bad = spark.createDataFrame([
    (None,                                    "OperationMode", "140673", 5),   # NULL event_hour
    (datetime.datetime(2024, 8, 15, 10, 0),   "AlarmCreated",  "140674", -1),  # negative count
    (datetime.datetime(2024, 8, 15, 11, 0),   "OperationMode", "140675", 3),
], schema=bad_schema)

print("Staged data (before quality check):")
new_gold_bad.show()

In [ ]:
# Quality gate: audit the staged DataFrame, publish only if checks pass
def audit_and_publish(spark, staged_df, target_table):
    """Run quality checks on staged_df. If all pass, append to target_table. Return True on publish."""
    failures = []

    null_count = staged_df.filter(F.col("event_hour").isNull()).count()
    if null_count > 0:
        failures.append(f"NULL event_hour: {null_count} rows")

    neg_count = staged_df.filter(F.col("event_count") < 0).count()
    if neg_count > 0:
        failures.append(f"Negative event_count: {neg_count} rows")

    if failures:
        print("Quality checks FAILED — data NOT written to gold:")
        for f in failures:
            print(f"  - {f}")
        return False
    else:
        staged_df.writeTo(target_table).append()
        print("Quality checks passed — data published to gold.")
        return True

# Should fail: bad data is discarded
audit_and_publish(spark, new_gold_bad, "local.gold.device_hourly")

# Verify: main table unchanged
main_count = spark.table("local.gold.device_hourly").count()
print(f"\nGold rows after failed WAP: {main_count:,} (original: {hourly_gold.count():,})")
print(f"Main unchanged: {main_count == hourly_gold.count()}")

In [ ]:
# Now publish clean data: fix the bad rows and run the gate again
new_gold_clean = spark.createDataFrame([
    (datetime.datetime(2024, 8, 15, 10, 0), "OperationMode", "140673", 5),
    (datetime.datetime(2024, 8, 15, 10, 0), "AlarmCreated",  "140674", 1),
    (datetime.datetime(2024, 8, 15, 11, 0), "OperationMode", "140675", 3),
], schema=bad_schema)

audit_and_publish(spark, new_gold_clean, "local.gold.device_hourly")

new_count = spark.table("local.gold.device_hourly").count()
print(f"\nGold rows after successful WAP: {new_count:,}")

## Data Lineage

Lineage means knowing where a gold row came from — which silver snapshots produced it.

A simple approach: add a `source_snapshot_id` column to gold tables. When debugging a suspicious aggregate, query `WHERE source_snapshot_id = X` to find the silver snapshot, then use PyIceberg time travel to inspect the raw events that contributed to it.

This creates a traceable path from a gold aggregate back to the individual silver records, without storing redundant copies of the raw data.

In [ ]:
# Build gold with lineage column
snapshot_id = silver_events.current_snapshot().snapshot_id

gold_with_lineage = silver_df.groupBy("event_hour", "type").agg(
    F.count("*").alias("event_count"),
    F.lit(snapshot_id).cast("long").alias("source_snapshot_id"),
    F.current_timestamp().alias("computed_at"),
)

gold_with_lineage.writeTo("local.gold.hourly_with_lineage").createOrReplace()
print("Gold with lineage:")
spark.sql("SELECT * FROM local.gold.hourly_with_lineage ORDER BY event_hour DESC LIMIT 5").show()

## Review Questions

1. At which medallion layer should schema validation be strictly enforced? Why?
2. What is the fundamental difference between a null check and a referential integrity check?
3. How does the Write-Audit-Publish pattern differ from checking data quality after a commit?
4. Why does adding `source_snapshot_id` to gold tables simplify debugging?
5. Your quality check for a gold table fails at 02:00 (low traffic hour). How do you investigate whether the failure is a real data issue or a normal low-traffic period?

## Challenges

- Introduce a deliberate duplicate: append the same 1,000 events twice to silver, recompute gold, and detect the duplication with a quality check.
- Build a `QualityGate` class that wraps a list of check functions and runs them in sequence, collecting all failures before raising (fail-all rather than fail-fast).
- Implement a clean WAP flow with good data: write to branch, pass all checks, fast-forward main, verify main now has the new data.
- Add a `check_freshness` function that raises if the max `event_hour` in the gold table is more than N hours old.

## Summary

Key takeaways:

- Data quality checks are SQL assertions: zero-row results = pass. Encode them as functions that raise on failure.
- The five quality dimensions for IoT gold: completeness (no nulls on key fields), accuracy (counts ≥ 0, timestamps plausible), consistency (referential integrity), uniqueness (no duplicate events), timeliness (freshness).
- Write-Audit-Publish uses Iceberg branches: write to a branch, audit there, drop (on failure) or fast-forward main (on success). Production consumers never see uncommitted data.
- Lineage columns (`source_snapshot_id`) create a traceable path from a gold row back to the silver snapshot that produced it.
- Next: structuring gold data for ML feature stores and LLM context.